# Supplementary Tables: Ensembl × CAT Gene Concordance

Generates two supplementary TSV tables for publication.

| Table | File | Content |
|-------|------|---------|
| S1 | supp_table_s1_gene_concordance_summary.tsv | Per-gene summary across all 462 assemblies |
| S2 | supp_table_s2_per_assembly_gene_pairs.tsv | Per-assembly × gene-pair detail |

**Input:** Pipeline output directory (set OUTPUT_DIR below)

In [1]:
!pip install -U pandas pyarrow

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import os
import time
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR = Path(os.getenv('HPRC_QC_OUTPUT_DIR',
    '/hps/nobackup/flicek/ensembl/genebuild/jackt/hprc/hprc-qc/results'))

QC_DIR = OUTPUT_DIR / 'qc_metrics'
RESULTS_DIR = OUTPUT_DIR / 'results'
SUPP_DIR = OUTPUT_DIR / 'supplementary_tables'
SUPP_DIR.mkdir(parents=True, exist_ok=True)

# Parquet caches — dramatically faster than re-reading 462 TSV files each run.
# Set FORCE_RELOAD=True to regenerate from raw files (e.g. after pipeline re-run).
CACHE_DIR = SUPP_DIR / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FORCE_RELOAD = False

print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"QC_DIR exists: {QC_DIR.exists()}")
print(f"RESULTS_DIR exists: {RESULTS_DIR.exists()}")
print(f"CACHE_DIR: {CACHE_DIR}")
print(f"FORCE_RELOAD: {FORCE_RELOAD}")

OUTPUT_DIR: /hps/nobackup/flicek/ensembl/genebuild/jackt/hprc/hprc-qc/results
QC_DIR exists: True
RESULTS_DIR exists: True
CACHE_DIR: /hps/nobackup/flicek/ensembl/genebuild/jackt/hprc/hprc-qc/results/supplementary_tables/cache
FORCE_RELOAD: False


## Load per-assembly data

In [3]:
_cache = CACHE_DIR / 'transcript_concordance.parquet'
if _cache.exists() and not FORCE_RELOAD:
    t0 = time.time()
    transcript_concordance_df = pd.read_parquet(_cache)
    print(f"Loaded transcript concordance from cache: {len(transcript_concordance_df):,} rows ({time.time()-t0:.1f}s)")
else:
    transcript_concordance_frames = []
    for i, accession_dir in enumerate(sorted(QC_DIR.iterdir())):
        if not accession_dir.is_dir():
            continue
        accession = accession_dir.name
        fp = accession_dir / f'{accession}_transcript_concordance.tsv'
        try:
            df = pd.read_csv(fp, sep='\t')
            if df.empty:
                print(f"WARNING: empty file {fp}")
                continue
            transcript_concordance_frames.append(df)
        except FileNotFoundError:
            print(f"WARNING: missing file {fp}")
        except Exception as e:
            print(f"WARNING: could not read {fp}: {e}")
        if (i + 1) % 50 == 0:
            print(f"  transcript concordance: {i+1} assemblies loaded...")

    transcript_concordance_df = pd.concat(transcript_concordance_frames, ignore_index=True)
    transcript_concordance_df.to_parquet(_cache, index=False)
    print(f"Loaded transcript concordance: {len(transcript_concordance_df):,} rows from {len(transcript_concordance_frames)} assemblies → cached")

Loaded transcript concordance from cache: 34,967,917 rows (10.6s)


In [4]:
_cache = CACHE_DIR / 'coding_integrity.parquet'
if _cache.exists() and not FORCE_RELOAD:
    t0 = time.time()
    coding_integrity_df = pd.read_parquet(_cache)
    print(f"Loaded coding integrity from cache: {len(coding_integrity_df):,} rows ({time.time()-t0:.1f}s)")
else:
    coding_integrity_frames = []
    for i, accession_dir in enumerate(sorted(QC_DIR.iterdir())):
        if not accession_dir.is_dir():
            continue
        accession = accession_dir.name
        fp = accession_dir / f'{accession}_coding_integrity.tsv'
        try:
            df = pd.read_csv(fp, sep='\t')
            if df.empty:
                print(f"WARNING: empty file {fp}")
                continue
            coding_integrity_frames.append(df)
        except FileNotFoundError:
            print(f"WARNING: missing file {fp}")
        except Exception as e:
            print(f"WARNING: could not read {fp}: {e}")
        if (i + 1) % 50 == 0:
            print(f"  coding integrity: {i+1} assemblies loaded...")

    coding_integrity_df = pd.concat(coding_integrity_frames, ignore_index=True)
    coding_integrity_df.to_parquet(_cache, index=False)
    print(f"Loaded coding integrity: {len(coding_integrity_df):,} rows from {len(coding_integrity_frames)} assemblies → cached")

Loaded coding integrity from cache: 8,649,784 rows (1.9s)


In [ ]:
import gc

# Gene presence files are not used — ensembl_gene_id and gene_name in those files
# are assembly-specific CAT IDs, not canonical. Instead we derive what we need from
# transcript_concordance_df (already loaded) and rbh_df (loaded in cell-8).
#
# gene_name / ensembl_name_map: built in cell-8 from rbh ensembl_name column
# gp_agg:  built here from transcript_concordance_df

_gp_cache = CACHE_DIR / 'gp_agg_v2.parquet'

def safe_mode(series):
    m = series.dropna().mode()
    return m.iloc[0] if len(m) > 0 else np.nan

if _gp_cache.exists() and not FORCE_RELOAD:
    t0 = time.time()
    gp_agg = pd.read_parquet(_gp_cache)
    print(f"Loaded gp_agg from cache: {len(gp_agg):,} genes ({time.time()-t0:.1f}s)")
else:
    t0 = time.time()
    n_assemblies_total = transcript_concordance_df['assembly_accession'].nunique()
    print(f"Building gp_agg from transcript concordance ({n_assemblies_total} assemblies)...")

    # Each TC row = Ensembl gene overlapped a CAT gene in that assembly → both present
    gp_agg = transcript_concordance_df.groupby('ensembl_gene_id').agg(
        n_assemblies_both=('assembly_accession', 'nunique'),
        ensembl_biotype=('ensembl_biotype', safe_mode),
    ).reset_index()
    gp_agg['n_assemblies_assessed'] = gp_agg['n_assemblies_both']
    gp_agg['pct_assemblies_both'] = (
        gp_agg['n_assemblies_both'] / n_assemblies_total * 100
    ).round(1)

    gp_agg.to_parquet(_gp_cache, index=False)
    print(f"Built gp_agg: {len(gp_agg):,} genes ({time.time()-t0:.1f}s) → cached")

print(f"gp_agg: {len(gp_agg):,} genes")
# ensembl_name_map and gp_name_series are built in cell-8 after rbh_df is loaded

In [6]:
_cache = CACHE_DIR / 'divergence.parquet'
if _cache.exists() and not FORCE_RELOAD:
    t0 = time.time()
    divergence_df = pd.read_parquet(_cache)
    print(f"Loaded grch38 divergence from cache: {len(divergence_df):,} rows ({time.time()-t0:.1f}s)")
else:
    divergence_frames = []
    for i, accession_dir in enumerate(sorted(QC_DIR.iterdir())):
        if not accession_dir.is_dir():
            continue
        accession = accession_dir.name
        fp = accession_dir / f'{accession}_grch38_divergence.tsv'
        if not fp.exists():
            continue
        try:
            df = pd.read_csv(fp, sep='\t')
            if df.empty:
                print(f"WARNING: empty file {fp}")
                continue
            divergence_frames.append(df)
        except Exception as e:
            print(f"WARNING: could not read {fp}: {e}")
        if (i + 1) % 50 == 0:
            print(f"  divergence: {len(divergence_frames)} assemblies loaded...")

    if divergence_frames:
        divergence_df = pd.concat(divergence_frames, ignore_index=True)
        divergence_df.to_parquet(_cache, index=False)
        print(f"Loaded grch38 divergence: {len(divergence_df):,} rows from {len(divergence_frames)} assemblies → cached")
    else:
        divergence_df = pd.DataFrame(columns=['assembly_accession', 'sample_name', 'ensembl_gene_id',
                                               'cat_gene_id', 'gene_name', 'ensembl_biotype',
                                               'ref_biotype', 'divergence_category'])
        print("WARNING: no grch38 divergence files found; divergence_df is empty")

Loaded grch38 divergence from cache: 18,562,662 rows (18.0s)


## Build Supplementary Table S2: Per-assembly gene-pair detail

In [ ]:
t0 = time.time()

# Load all RBH gene pair files
# rbh files use ensembl_id/cat_id column names and have no assembly_accession —
# add it from the directory name and rename to match pipeline-wide conventions.
rbh_frames = []
for i, accession_dir in enumerate(sorted(RESULTS_DIR.iterdir())):
    if not accession_dir.is_dir():
        continue
    accession = accession_dir.name
    fp = accession_dir / f'{accession}.gene_pairs_rbh.tsv'
    try:
        df = pd.read_csv(fp, sep='\t')
        if df.empty:
            print(f"WARNING: empty file {fp}")
            continue
        df['assembly_accession'] = accession
        df = df.rename(columns={'ensembl_id': 'ensembl_gene_id', 'cat_id': 'cat_gene_id'})
        rbh_frames.append(df)
    except FileNotFoundError:
        print(f"WARNING: missing file {fp}")
    except Exception as e:
        print(f"WARNING: could not read {fp}: {e}")
    if (i + 1) % 50 == 0:
        print(f"  RBH pairs: {len(rbh_frames)} assemblies loaded...")

rbh_df = pd.concat(rbh_frames, ignore_index=True)
del rbh_frames
gc.collect()
print(f"Loaded RBH gene pairs: {len(rbh_df):,} rows from {rbh_df['assembly_accession'].nunique()} assemblies ({time.time()-t0:.1f}s)")

# Keep only RBH pairs
rbh_df = rbh_df[rbh_df['is_rbh'] == True].copy()
print(f"After filtering is_rbh=True: {len(rbh_df):,} rows")

# Build ensembl_name_map (ensembl_gene_id → gene_name) from rbh ensembl_name column.
# This is the canonical HGNC symbol and is consistent across assemblies.
_nm_cache = CACHE_DIR / 'ensembl_name_map_v2.parquet'
if _nm_cache.exists() and not FORCE_RELOAD:
    ensembl_name_map = pd.read_parquet(_nm_cache)
    print(f"Loaded ensembl_name_map from cache: {len(ensembl_name_map):,} entries")
else:
    ensembl_name_map = (
        rbh_df[['ensembl_gene_id', 'ensembl_name']]
        .rename(columns={'ensembl_name': 'gene_name'})
        .dropna(subset=['ensembl_gene_id', 'gene_name'])
        .drop_duplicates('ensembl_gene_id')
        .reset_index(drop=True)
    )
    ensembl_name_map.to_parquet(_nm_cache, index=False)
    print(f"Built ensembl_name_map: {len(ensembl_name_map):,} entries → cached")

gp_name_series = ensembl_name_map.set_index('ensembl_gene_id')['gene_name']

# Add gene_name to gp_agg
gp_agg['gene_name'] = gp_agg['ensembl_gene_id'].map(gp_name_series)

# --- Build S2 table ---

# Join transcript concordance onto rbh_df via (assembly_accession, ensembl_gene_id, cat_gene_id)
join_cols = ['assembly_accession', 'ensembl_gene_id', 'cat_gene_id']
tc_cols = join_cols + [
    'n_ensembl_transcripts', 'n_cat_transcripts',
    'n_ens_exact', 'n_cat_exact',
    'ens_to_cat_concordance_rate', 'cat_to_ens_concordance_rate',
    'avg_jaccard_index'
]
tc_subset = transcript_concordance_df[tc_cols].drop_duplicates(subset=join_cols)
merged = rbh_df.merge(tc_subset, on=join_cols, how='left')
print(f"After joining transcript concordance: {len(merged):,} rows")

# gene_name: rbh file already has ensembl_name; also map via gp_name_series for consistency
merged['gene_name'] = merged['ensembl_name']

# Join coding integrity
ci_cols = join_cols + ['classification', 'start_codon_match', 'stop_codon_match', 'frameshift_detected']
ci_subset = coding_integrity_df[ci_cols].drop_duplicates(subset=join_cols)
ci_subset = ci_subset.rename(columns={'classification': 'cds_classification'})
merged = merged.merge(ci_subset, on=join_cols, how='left')
print(f"After joining coding integrity: {len(merged):,} rows")

# Join grch38 divergence
div_cols = join_cols + ['divergence_category']
if not divergence_df.empty:
    div_subset = divergence_df[div_cols].drop_duplicates(subset=join_cols)
    merged = merged.merge(div_subset, on=join_cols, how='left')
else:
    merged['divergence_category'] = None
print(f"After joining divergence: {len(merged):,} rows")

# Compute exact pct columns (handle div by zero)
merged['ens_to_cat_exact_pct'] = np.where(
    merged['n_ensembl_transcripts'] > 0,
    (merged['n_ens_exact'] / merged['n_ensembl_transcripts'] * 100).round(1),
    np.nan
)
merged['cat_to_ens_exact_pct'] = np.where(
    merged['n_cat_transcripts'] > 0,
    (merged['n_cat_exact'] / merged['n_cat_transcripts'] * 100).round(1),
    np.nan
)

# Select and order final columns
s2_cols = [
    'assembly_accession',
    'sample_name',
    'ensembl_gene_id',
    'cat_gene_id',
    'gene_name',
    'ensembl_biotype',
    'cat_biotype',
    'frac_ensembl_covered',
    'frac_cat_covered',
    'n_ensembl_transcripts',
    'n_cat_transcripts',
    'n_ens_exact',
    'n_cat_exact',
    'ens_to_cat_exact_pct',
    'cat_to_ens_exact_pct',
    'ens_to_cat_concordance_rate',
    'cat_to_ens_concordance_rate',
    'avg_jaccard_index',
    'cds_classification',
    'start_codon_match',
    'stop_codon_match',
    'frameshift_detected',
    'divergence_category',
]
s2_cols = [c for c in s2_cols if c in merged.columns]
s2_df = merged[s2_cols].copy()

out_s2 = SUPP_DIR / 'supp_table_s2_per_assembly_gene_pairs.tsv'
s2_df.to_csv(out_s2, sep='\t', index=False)
print(f"\nSaved S2: {out_s2}")
print(f"Shape: {s2_df.shape} ({time.time()-t0:.1f}s total)")
s2_df.head(3)

## Build Supplementary Table S1: Gene-level concordance summary

In [ ]:
# gp_agg, ensembl_name_map, gp_name_series, and safe_mode are pre-computed in the gene presence cell.
# ensembl_name_map (ensembl_gene_id → gene_name) is used for all joins — ~60K rows, not 35M.

t0 = time.time()

# --- Transcript concordance medians ---
# Join gene_name via ensembl_gene_id only (compact map, avoids 35M-row merge)
tc_with_gene = transcript_concordance_df.merge(ensembl_name_map, on='ensembl_gene_id', how='left')
print(f"tc_with_gene: {len(tc_with_gene):,} rows ({time.time()-t0:.1f}s)")

tc_with_gene['ens_exact_pct'] = np.where(
    tc_with_gene['n_ensembl_transcripts'] > 0,
    tc_with_gene['n_ens_exact'] / tc_with_gene['n_ensembl_transcripts'] * 100,
    np.nan
)
tc_with_gene['cat_exact_pct'] = np.where(
    tc_with_gene['n_cat_transcripts'] > 0,
    tc_with_gene['n_cat_exact'] / tc_with_gene['n_cat_transcripts'] * 100,
    np.nan
)

tc_medians = tc_with_gene.dropna(subset=['ensembl_gene_id']).groupby('ensembl_gene_id').agg(
    median_ens_to_cat_exact_pct=('ens_exact_pct', 'median'),
    median_cat_to_ens_exact_pct=('cat_exact_pct', 'median'),
    median_ens_to_cat_concordance_rate=('ens_to_cat_concordance_rate', 'median'),
    median_cat_to_ens_concordance_rate=('cat_to_ens_concordance_rate', 'median'),
).reset_index().round(1)
del tc_with_gene
gc.collect()
print(f"tc_medians: {len(tc_medians):,} genes ({time.time()-t0:.1f}s)")

# --- CDS integrity ---
ci_with_gene = coding_integrity_df.merge(ensembl_name_map, on='ensembl_gene_id', how='left')

ci_agg = ci_with_gene.dropna(subset=['ensembl_gene_id']).groupby('ensembl_gene_id').agg(
    n_assemblies_cds_assessed=('assembly_accession', 'nunique'),
).reset_index()

ci_bool = ci_with_gene.dropna(subset=['ensembl_gene_id']).copy()
del ci_with_gene
gc.collect()

for col in ['start_codon_match', 'stop_codon_match', 'frameshift_detected']:
    if col in ci_bool.columns:
        ci_bool[col] = ci_bool[col].map({True: True, False: False,
                                          'True': True, 'False': False,
                                          1: True, 0: False})

ci_pct = ci_bool.groupby('ensembl_gene_id').agg(
    _n_start=('start_codon_match', 'count'),
    _sum_start=('start_codon_match', 'sum'),
    _n_stop=('stop_codon_match', 'count'),
    _sum_stop=('stop_codon_match', 'sum'),
    _n_fs=('frameshift_detected', 'count'),
    _sum_fs=('frameshift_detected', 'sum'),
).reset_index()
del ci_bool
gc.collect()

ci_pct['pct_start_codon_match'] = np.where(
    ci_pct['_n_start'] > 0, (ci_pct['_sum_start'] / ci_pct['_n_start'] * 100).round(1), np.nan)
ci_pct['pct_stop_codon_match'] = np.where(
    ci_pct['_n_stop'] > 0, (ci_pct['_sum_stop'] / ci_pct['_n_stop'] * 100).round(1), np.nan)
ci_pct['pct_frameshift_detected'] = np.where(
    ci_pct['_n_fs'] > 0, (ci_pct['_sum_fs'] / ci_pct['_n_fs'] * 100).round(1), np.nan)
ci_pct = ci_pct[['ensembl_gene_id', 'pct_start_codon_match', 'pct_stop_codon_match', 'pct_frameshift_detected']]
ci_agg = ci_agg.merge(ci_pct, on='ensembl_gene_id', how='left')
print(f"ci_agg: {len(ci_agg):,} genes ({time.time()-t0:.1f}s)")

# --- Divergence ---
if not divergence_df.empty and 'ensembl_gene_id' in divergence_df.columns:
    div_agg_base = divergence_df.dropna(subset=['ensembl_gene_id', 'divergence_category'])
    div_counts = (div_agg_base.groupby(['ensembl_gene_id', 'divergence_category'])
                  .size().unstack(fill_value=0).reset_index())
    for cat in ['both_agree_reference', 'both_agree_diverged', 'ensembl_specific', 'cat_specific']:
        if cat not in div_counts.columns:
            div_counts[cat] = 0
    div_counts['_n_div_total'] = div_counts[['both_agree_reference', 'both_agree_diverged',
                                              'ensembl_specific', 'cat_specific']].sum(axis=1)
    for cat in ['both_agree_reference', 'both_agree_diverged', 'ensembl_specific', 'cat_specific']:
        div_counts[f'pct_{cat}'] = np.where(
            div_counts['_n_div_total'] > 0,
            (div_counts[cat] / div_counts['_n_div_total'] * 100).round(1), np.nan)
    div_mode = (div_agg_base.groupby('ensembl_gene_id')['divergence_category']
                .agg(safe_mode).reset_index())
    div_mode.columns = ['ensembl_gene_id', 'predominant_divergence_category']
    div_final = div_counts[['ensembl_gene_id', 'pct_both_agree_reference', 'pct_both_agree_diverged',
                              'pct_ensembl_specific', 'pct_cat_specific']]
    div_final = div_final.merge(div_mode, on='ensembl_gene_id', how='left')
else:
    div_final = pd.DataFrame(columns=['ensembl_gene_id', 'predominant_divergence_category',
                                       'pct_both_agree_reference', 'pct_both_agree_diverged',
                                       'pct_ensembl_specific', 'pct_cat_specific'])
print(f"div_final: {len(div_final):,} genes ({time.time()-t0:.1f}s)")

# --- Assemble S1 ---
# gp_agg columns: ensembl_gene_id, n_assemblies_*, pct_assemblies_both, ensembl_biotype, gene_name
s1_df = gp_agg.copy()
s1_df = s1_df.merge(tc_medians, on='ensembl_gene_id', how='left')
s1_df = s1_df.merge(ci_agg, on='ensembl_gene_id', how='left')
s1_df = s1_df.merge(div_final, on='ensembl_gene_id', how='left')
s1_df['predominant_divergence_category'] = s1_df['predominant_divergence_category'].fillna('N/A')

s1_col_order = [
    'ensembl_gene_id',
    'gene_name',
    'ensembl_biotype',
    'n_assemblies_assessed',
    'n_assemblies_both',
    'n_assemblies_ensembl_only',
    'n_assemblies_cat_only',
    'pct_assemblies_both',
    'median_ens_to_cat_exact_pct',
    'median_cat_to_ens_exact_pct',
    'median_ens_to_cat_concordance_rate',
    'median_cat_to_ens_concordance_rate',
    'n_assemblies_cds_assessed',
    'pct_start_codon_match',
    'pct_stop_codon_match',
    'pct_frameshift_detected',
    'predominant_divergence_category',
    'pct_both_agree_reference',
    'pct_both_agree_diverged',
    'pct_ensembl_specific',
    'pct_cat_specific',
]
s1_col_order = [c for c in s1_col_order if c in s1_df.columns]
s1_df = s1_df[s1_col_order].sort_values('ensembl_gene_id').reset_index(drop=True)

out_s1 = SUPP_DIR / 'supp_table_s1_gene_concordance_summary.tsv'
s1_df.to_csv(out_s1, sep='\t', index=False)
print(f"\nSaved S1: {out_s1}")
print(f"Shape: {s1_df.shape} ({time.time()-t0:.1f}s total)")
s1_df.head(3)

## Summary

In [ ]:
print(f"Table S1: {len(s1_df):,} genes \u00d7 {len(s1_df.columns)} columns")
print(f"Table S2: {len(s2_df):,} gene-pair records \u00d7 {len(s2_df.columns)} columns")
print(f"\nFiles written to: {SUPP_DIR}")
for f in sorted(SUPP_DIR.glob('*.tsv')):
    size = f.stat().st_size / 1024**2
    print(f"  {f.name}: {size:.1f} MB")